# Fake Jobs – Cleaned
- Numerische/enkodierte Features, Freitexte entfernt, skaliert
- Eine CSV ohne Split; Transforms auf dem gesamten Datensatz gefittet

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

## Daten laden
- Label `fraudulent`, stabile `row_id` aus `job_id`

In [2]:
df = pd.read_csv("../../data/raw/fake_job_postings.csv")
df["row_id"] = df["job_id"]
y = df["fraudulent"]
print(df.shape, "outlier rate:", round(y.mean(), 4))

(17880, 19) outlier rate: 0.0484


## Feature Engineering
- `location` → country/state/city
- `salary_range` → salary_avg + salary_missing

In [3]:
parts = df["location"].fillna("").str.split(",", n=2, expand=True).reindex(columns=[0, 1, 2])
df["country"] = parts[0].str.strip().replace("", "missing")
df["state"] = parts[1].fillna("").str.strip().replace("", "missing")
df["city"] = parts[2].fillna("").str.strip().replace("", "missing")

sal = df["salary_range"].fillna("").str.split("-", n=1, expand=True).reindex(columns=[0, 1])
df["salary_avg"] = pd.concat([pd.to_numeric(sal[0], errors="coerce"),
                              pd.to_numeric(sal[1], errors="coerce")], axis=1).mean(axis=1)

## Spalten droppen & Kategorien vorbereiten
- IDs, Roh-Strukturspalten und alle 5 Freitexte entfernen

In [4]:
df = df.drop(columns=["job_id", "location", "salary_range", "fraudulent",
                      "title", "company_profile", "description", "requirements", "benefits"])

cat_cols = ["industry", "function", "department", "country", "state", "city",
            "employment_type", "required_experience", "required_education"]
binary = ["telecommuting", "has_company_logo", "has_questions"]
for c in cat_cols:
    df[c] = df[c].fillna("missing").replace("", "missing")

## Encoding & Scaling (auf dem gesamten Datensatz)
- Alle Kategorien per Frequency-Encoding; numerische + freq-Spalten StandardScaler-skaliert (13 Features)

In [5]:
# numeric impute (median)
df["salary_avg"] = df["salary_avg"].fillna(df["salary_avg"].median())

# frequency encoding (alle Kategorien)
for c in cat_cols:
    df[c] = df[c].map(df[c].value_counts(normalize=True)).astype(float)

# scale numeric + frequency-encoded columns (13 Features)
num_cols = ["salary_avg"] + binary + cat_cols
scaler = StandardScaler()
df[num_cols] = scaler.fit_transform(df[num_cols])

## Zusammenbauen & speichern

In [6]:
keep = ["row_id"] + num_cols
out = df[keep].reset_index(drop=True)
out["fraudulent"] = y.values

out.to_csv("../../data/preprocessed/cleaned_fake_jobs.csv", index=False)

## Verifikation

In [7]:
print("shape", out.shape)
print("outlier rate", round(out["fraudulent"].mean(), 4))
assert out.drop(columns=["row_id"]).isna().sum().sum() == 0
print("row_id unique:", out["row_id"].is_unique)

shape (17880, 15)
outlier rate 0.0484
row_id unique: True
